# PLEASE CLONE THIS NOTEBOOK INTO YOUR PERSONAL FOLDER

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import matplotlib.pyplot as plt
print("Welcome to the W261 final project!") 


# Know your mount
Here is the mounting for this class, your source for the original data! Remember, you only have Read access, not Write! Also, become familiar with `dbutils` the equivalent of `gcp` in DataProc

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

In [0]:
dbutils.fs.help()

# Data for the Project

For the project you will have 4 sources of data:

1. Airlines Data: This is the raw data of flights information. You have 3 months, 6 months, 1 year, and full data from 2015 to 2019. Remember the maxima: "Test, Test, Test", so a lot of testing in smaller samples before scaling up! Location of the data? `dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data/`, `dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_1y/`, etc. (Below the dbutils to get the folders)
2. Weather Data: Raw data for weather information. Same as before, we are sharing 3 months, 6 months, 1 year
3. Stations data: Extra information of the location of the different weather stations. Location `dbfs:/mnt/mids-w261/datasets_final_project_2022/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/`
4. OTPW Data: This is our joined data (We joined Airlines and Weather). This is the main dataset for your project, the previous 3 are given for reference. You can attempt your own join for Extra Credit. Location `dbfs:/mnt/mids-w261/OTPW_60M/OTPW_60M/` and more, several samples are given!

In [0]:
# Airline Data    
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/")
display(df_flights)

In [0]:
# Weather data
df_weather = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")
display(df_weather)

In [0]:
# Stations data      
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
display(df_stations)

In [0]:
# OTPW
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")
display(df_otpw)


# Example of EDA

In [0]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt

#df_weather = spark.read.parquet(f"{data_BASE_DIR}/datasets_final_project_2022/parquet_weather_data_3m/")

# Weather Data
df_weather = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")

# Airport and Airport Codes Data
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
df_airport_codes =  spark.read.format("csv").option("header","true").load('dbfs:/mnt/mids-w261/airport-codes_csv.csv')

# Flights Data
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/")

# OTPW Data
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")

# Grouping and aggregation for df_stations
grouped_stations = df_stations.groupBy('neighbor_id').agg(
    F.avg('distance_to_neighbor').alias('avg_distance_to_neighbor'),
).orderBy('avg_distance_to_neighbor')

display(grouped_stations)

# Grouping and aggregation for df_flights
grouped_flights = df_flights.groupBy('OP_UNIQUE_CARRIER').agg(
    F.avg('DEP_DELAY').alias('Avg_DEP_DELAY'),
    F.avg('ARR_DELAY').alias('Avg_ARR_DELAY'),
    F.avg('DISTANCE').alias('Avg_DISTANCE')
)

display(grouped_flights)

# Convert columns to appropriate data types
#df_weather = df_weather.withColumn("HourlyPrecipitationDouble", F.col("HourlyPrecipitation").cast("double"))
#df_weather = df_weather.withColumn("HourlyVisibilityDouble", F.col("HourlyVisibility").cast("double"))
#df_weather = df_weather.withColumn("HourlyWindSpeedDouble", F.col("HourlyWindSpeed").cast("double")).filter(F.col("HourlyWindSpeedDouble") #< 2000)

#df_weather = df_weather.withColumn("HourlyWindSpeedDouble",F.expr("try_cast(HourlyWindSpeed as double)")).filter(F.col("HourlyWindSpeedDouble") < 2000)

df_weather = (
    df_weather
    .withColumn("HourlyPrecipitationDouble", F.expr("try_cast(HourlyPrecipitation as double)"))
    .withColumn("HourlyVisibilityDouble", F.expr("try_cast(HourlyVisibility as double)"))
    .withColumn("HourlyWindSpeedDouble", F.expr("try_cast(HourlyWindSpeed as double)"))
    .filter(F.col("HourlyWindSpeedDouble") < 2000)
)

# Overlayed boxplots for df_weather
weather_cols = ['HourlyPrecipitationDouble', 'HourlyVisibilityDouble', 'HourlyWindSpeedDouble']
weather_data = df_weather.select(*weather_cols).toPandas()

plt.figure(figsize=(10, 6))
weather_data.boxplot(column=weather_cols)
plt.title('Boxplots of Weather Variables')
plt.xlabel('Weather Variables')
plt.ylabel('Values')
plt.xticks(rotation=45)
plt.show()

# EDA - Airline Data


In [0]:
# Airport and Airport Codes Data
df_stations = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/")
df_airport_codes =  spark.read.format("csv").option("header","true").load('dbfs:/mnt/mids-w261/airport-codes_csv.csv')

In [0]:
display(df_stations)
df_stations.printSchema()
df_stations.show(5)
df_stations.count()

In [0]:
display(df_airport_codes)
df_airport_codes.printSchema()
df_airport_codes.show(5)
df_airport_codes.count()

In [0]:
grouped_stations = df_stations.groupBy('neighbor_id').agg(
    F.avg('distance_to_neighbor').alias('avg_distance_to_neighbor')
).orderBy('avg_distance_to_neighbor')

display(grouped_stations)

distance_data = grouped_stations.toPandas()

plt.figure(figsize=(8,5))
plt.hist(distance_data["avg_distance_to_neighbor"], bins=40)
plt.xlabel("Average Distance to Weather Station")
plt.ylabel("Number of Airports")
plt.title("Distribution of Weather Station Distance to Airports")
plt.show()

plt.figure(figsize=(6,5))
plt.boxplot(distance_data["avg_distance_to_neighbor"])
plt.ylabel("Average Station Distance")
plt.title("Boxplot of Airport–Weather Station Distance")
plt.show()

In [0]:
df_stations.select("distance_to_neighbor").describe().show()

df_airport_codes.groupBy("type").count().orderBy(F.desc("count")).show()

In [0]:
df_airport_codes.groupBy("iso_country").count().orderBy(F.desc("count")).show()

top_countries = (
    df_airport_codes
    .groupBy("iso_country")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
    .toPandas()
)

plt.figure(figsize=(10,6))
plt.bar(top_countries["iso_country"], top_countries["count"])

plt.title("Top 20 Countries by Number of Airports")
plt.xlabel("Country")
plt.ylabel("Number of Airports")

plt.show()

# Join Airport Codes with Stations
### This helps connect airports → weather stations.

In [0]:
airport_station = df_stations.join(
    df_airport_codes,
    df_stations.neighbor_call == df_airport_codes.ident,
    "left"
)

display(airport_station)

In [0]:
import pyspark.sql.functions as F

df_airport_codes = df_airport_codes.withColumn(
    "longitude",
    F.split(F.col("coordinates"), ",").getItem(0).cast("double")
).withColumn(
    "latitude",
    F.split(F.col("coordinates"), ",").getItem(1).cast("double")
)

df_airport_codes.select("ident","latitude","longitude").show(5)

In [0]:
airport_map = df_airport_codes.select("latitude","longitude").dropna().toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.scatter(
    airport_map["longitude"],
    airport_map["latitude"],
    s=2
)

plt.title("Geographic Distribution of Airports")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [0]:
stations_sample = df_stations.limit(2000).toPandas()

plt.figure(figsize=(10,6))

plt.scatter(
    stations_sample["neighbor_lon"],
    stations_sample["neighbor_lat"],
    s=5,
    label="Airports"
)

plt.scatter(
    stations_sample["lon"],
    stations_sample["lat"],
    s=3,
    label="Weather Stations"
)

plt.title("Airport and Weather Station Coverage")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()

In [0]:
distance_data = df_stations.select("distance_to_neighbor").toPandas()

plt.figure(figsize=(8,5))
plt.hist(distance_data["distance_to_neighbor"], bins=50)
plt.title("Distribution of Weather Station Distance to Airports")
plt.xlabel("Distance to Airport")
plt.ylabel("Frequency")
plt.show()

In [0]:
df_airport_codes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_airport_codes.columns
]).show()

In [0]:
df_airport_codes.filter(
    F.col("type") == "large_airport"
).select("ident","name","municipality").show()

# Exploratory Data Analysis (EDA): Stations and Airport Codes Datasets

## 1. Objective
The objective of this exploratory data analysis (EDA) was to understand the structure and characteristics of two supporting datasets used in the airline delay prediction project:

- **`df_stations`**: Contains mappings between weather stations and nearby airports, including the distance between them.
- **`df_airport_codes`**: Provides airport metadata such as airport type, geographic location, and identification codes.

These datasets are important because they allow us to connect **flight operations with nearby weather observations**, which is critical for building features related to weather impacts on flight delays.

---

# 2. Stations Dataset Overview

The **stations dataset** was loaded from a parquet source and inspected using schema and sample displays.

Key fields include:

| Column | Description |
|------|-------------|
| `station_id` | Weather station identifier |
| `neighbor_id` | Nearby airport identifier |
| `neighbor_call` | Airport call code |
| `neighbor_name` | Airport name |
| `neighbor_state` | Airport state |
| `distance_to_neighbor` | Distance between station and airport |

Each weather station is linked to a nearby airport, enabling the integration of weather observations into airport-level flight data.

---

## Distance Statistics

Descriptive statistics for the distance between stations and airports show:

| Metric | Value |
|------|------|
| Count | 5,004,169 |
| Mean | 1343.52 |
| Std Dev | 948.85 |
| Min | 0 |
| Max | 6435.97 |

### Interpretation

- The dataset contains **over 5 million station-airport relationships**, indicating extensive coverage.
- The **average distance between a weather station and its associated airport is approximately 1343 units**.
- Some stations are located **directly at airports (distance = 0)**.
- Other stations are **much farther away**, suggesting that weather conditions may not always perfectly represent airport conditions.

This variability should be considered when incorporating weather data into delay prediction models.

---

# 3. Airport Codes Dataset Overview

The **airport codes dataset** contains metadata describing airports around the world.

Key attributes include:

| Column | Description |
|------|-------------|
| `ident` | Airport identifier |
| `type` | Airport type |
| `name` | Airport name |
| `elevation_ft` | Elevation in feet |
| `iso_country` | Country code |
| `iso_region` | Regional code |
| `municipality` | City |
| `gps_code` | GPS airport code |
| `iata_code` | IATA airport code |
| `coordinates` | Geographic coordinates |

This dataset provides important contextual information about airports that can enrich the flight dataset.

---

# 4. Airport Type Distribution

Airport types were analyzed to understand the structure of the dataset.

| Airport Type | Count |
|-------------|-------|
| small_airport | 34,808 |
| heliport | 12,028 |
| medium_airport | 4,537 |
| closed | 4,378 |
| seaplane_base | 1,030 |
| large_airport | 616 |

### Interpretation

- The majority of entries correspond to **small airports or heliports**.
- Only a small portion represent **large commercial airports**.
- Since the airline dataset focuses on commercial flights, **large and medium airports are likely the most relevant subset for analysis**.

---

# 5. Country Distribution

The dataset contains airports from many countries. The top countries include:

| Country | Airport Count |
|-------|------|
| United States | 23,256 |
| Brazil | 5,043 |
| Canada | 2,794 |
| Australia | 2,018 |
| Mexico | 1,405 |

### Interpretation

The airport dataset has **global coverage**, but the **United States dominates**, which aligns well with the flight dataset used in this project.

---

# 6. Linking Stations with Airports

The stations dataset was joined with airport metadata using the airport identifier:


This join enriches weather stations with airport information such as:

- airport type
- location
- country
- municipality

This connection enables weather conditions recorded at stations to be associated with specific airports.

---

# 7. Missing Value Analysis

A missing-value check was performed on the airport metadata dataset.

Notable missing values include:

| Column | Missing Values |
|------|------|
| elevation_ft | 7,813 |
| municipality | 5,894 |
| gps_code | 15,860 |
| iata_code | 48,196 |
| local_code | 27,391 |

### Interpretation

- Some airport metadata fields have **significant missingness**.
- The `iata_code` field has the largest number of missing values.
- This is expected because many entries correspond to small or inactive airports that do not have commercial airline codes.

For joining datasets, **`ident` appears to be the most reliable key**.

---

# 8. Key Findings

The EDA produced several important insights:

1. The stations dataset contains **extensive mappings between airports and weather stations**.
2. Weather stations vary widely in their **distance from airports**, which may affect weather accuracy.
3. The airport metadata dataset includes **global airport information**, but most relevant records are in the United States.
4. The majority of airports in the dataset are **small airports or heliports**, while only a small subset represent commercial airports.
5. Several metadata fields contain **substantial missing values**, particularly `iata_code`.

---

# 9. Implications for Modeling

These findings inform several decisions for the modeling phase:

- Weather data should be carefully linked to airports, considering **station distance as a potential feature or filter**.
- The airport metadata dataset should likely be **filtered to medium and large airports** for airline analysis.
- **Airport identifier (`ident`) should be used as the primary join key** rather than `iata_code`.
- Additional validation may be needed to ensure that weather stations appropriately represent airport conditions.

---

# 10. Next Steps

Further analysis will include:

- Filtering airport metadata to **relevant airport types**.
- Evaluating how well airport identifiers match those in the **flight dataset**.
- Integrating **weather, flight, and airport datasets** to build predictive features for flight delay modeling.



## EDA for Stations and Airport Codes Datasets

### 1. Objective
The objective of this exploratory data analysis (EDA) was to understand the structure and characteristics of two supporting datasets used in the airline delay prediction project:

- **`df_stations`**: Contains mappings between weather stations and nearby airports, including the distance between them.
- **`df_airport_codes`**: Provides airport metadata such as airport type, geographic location, and identification codes.

These datasets are important because they allow us to connect **flight operations with nearby weather observations**, which is critical for building features related to weather impacts on flight delays.

---

### 2. Stations Dataset Overview

The **stations dataset** was loaded from a parquet source and inspected using schema exploration and descriptive statistics.

Key fields include:

| Column | Description |
|------|-------------|
| `station_id` | Weather station identifier |
| `neighbor_id` | Nearby airport identifier |
| `neighbor_call` | Airport call code |
| `neighbor_name` | Airport name |
| `neighbor_state` | Airport state |
| `distance_to_neighbor` | Distance between station and airport |

Each record represents a relationship between a **weather station and a nearby airport**.

---

#### Distance Statistics

Summary statistics for `distance_to_neighbor` show:

| Metric | Value |
|------|------|
| Count | ~5,004,169 |
| Mean | ~1343 |
| Std Dev | ~948 |
| Min | 0 |
| Max | ~6435 |

#### Interpretation

- The dataset contains **over 5 million station–airport relationships**, indicating extensive weather coverage.
- Some stations are located **directly at airports (distance = 0)**.
- Other stations are **much farther away**, which may reduce the representativeness of the weather data.
- The large variation suggests **distance to the weather station may influence the accuracy of weather features** used in modeling.

---

### 3. Airport Codes Dataset Overview

The **airport codes dataset** provides metadata describing airports around the world.

Key attributes include:

| Column | Description |
|------|-------------|
| `ident` | Airport identifier |
| `type` | Airport type |
| `name` | Airport name |
| `elevation_ft` | Airport elevation |
| `iso_country` | Country code |
| `iso_region` | Regional code |
| `municipality` | City |
| `gps_code` | GPS airport code |
| `iata_code` | IATA airport code |
| `coordinates` | Geographic coordinates |

The `coordinates` column contains latitude and longitude values stored as a comma-separated string. These were split to create separate **latitude and longitude variables**, enabling geographic visualization of airport locations.

---

### 4. Airport Type Distribution

Airport types in the dataset include:

| Airport Type | Count |
|-------------|------|
| small_airport | ~34,800 |
| heliport | ~12,000 |
| medium_airport | ~4,500 |
| closed | ~4,300 |
| seaplane_base | ~1,000 |
| large_airport | ~600 |

##### Interpretation

- Most entries correspond to **small airports or heliports**.
- Only a small portion correspond to **large commercial airports**.
- Since the airline dataset focuses on scheduled flights, **medium and large airports are the most relevant subset for modeling**.

---

### 5. Geographic Distribution of Airports

To understand the geographic coverage of the airport dataset, the number of airports per country was analyzed.

![](images/distribution_of_weather_station_distance_to_airports.png)

Top countries include:

| Country | Airport Count |
|-------|------|
| United States | ~23,256 |
| Brazil | ~5,043 |
| Canada | ~2,794 |
| Australia | ~2,018 |
| Mexico | ~1,405 |

##### Visualization Insights
A bar chart visualization shows that the airport dataset has **global coverage but is heavily dominated by the United States**, which aligns well with the airline dataset used in this project.

---

### 6. Geographic Airport Map

After extracting latitude and longitude from the `coordinates` column, a scatter plot map was created to visualize the **spatial distribution of airports worldwide**.

![](images/geographic_distribution_of_airports.png)

##### Interpretation

The map reveals several key patterns:

- High density of airports in **North America**, particularly in the United States.
- Moderate airport density across **Europe and South America**.
- Sparse coverage in **remote or low-population regions**.

This confirms that the dataset has **broad global coverage**, while still being heavily concentrated in regions with large aviation networks.

---

### 7. Linking Stations with Airports

To connect weather data with airport metadata, the stations dataset was joined with the airport codes dataset using the airport identifier:

df_stations.neighbor_call == df_airport_codes.ident


This join enriches the station dataset with additional airport attributes such as:

- airport type
- country
- geographic location
- municipality

This linkage enables weather observations recorded at stations to be associated with specific airports in the flight dataset.

---

### 8. Missing Value Analysis

A missing value analysis revealed that several metadata fields contain incomplete data.

| Column | Missing Values |
|------|------|
| elevation_ft | moderate |
| municipality | moderate |
| gps_code | high |
| iata_code | very high |
| local_code | high |

##### Interpretation

- Many airports in the dataset do not have **IATA codes**, since they are not commercial airports.
- The **`ident` column is the most reliable identifier for joining datasets**.
- Missing values in other metadata fields are less critical for the modeling objective.

---

### 9. Key Findings

The exploratory analysis produced several important insights:

1. The stations dataset provides **extensive coverage linking airports and weather stations**.
2. Weather stations vary significantly in **distance from airports**, which may influence weather feature accuracy.
3. The airport codes dataset provides **global airport coverage**, but is strongly dominated by the United States.
4. Most airports in the dataset are **small airports or heliports**, with relatively few large commercial airports.
5. Geographic visualization confirms that airport density aligns with major aviation regions.
6. The **`ident` field is the most reliable join key** for integrating airport metadata with other datasets.

---

### 10. Implications for Modeling

These findings inform several design choices for the modeling phase:

- Weather observations should be linked to airports carefully, considering **station distance as a potential feature or filtering criterion**.
- Airport metadata should likely be **filtered to medium and large airports** when focusing on commercial flight operations.
- The **`ident` airport identifier should be used for dataset joins** rather than `iata_code`.
- Geographic features such as airport location may be useful for modeling regional delay patterns.

---

### 11. Next Steps

Further analysis will focus on:

- Filtering airport metadata to focus on **relevant commercial airports**.
- Validating joins between **flights, airports, and weather stations**.
- Integrating **weather and airport metadata into the flight dataset**.
- Performing additional EDA on **flight delays and weather impacts** to support predictive modeling.